# 01 · Обзор журналов СМВУ: объёмы, схема, сводка по годам

**Цель** — зафиксировать общую картину: сколько событий, каналов, тревог по годам, как менялась схема и покрытие. Это базовый артефакт для всех дальнейших решений (объёмы выборки, временной сплит, баланс классов).

**Источник:** `dataset/ext-journal-{2019..2026}.csv` (распакованные из `.7z`). Каждый архив содержит ровно один CSV с одинаковой схемой.

Схема журнала: `ид_события, ид_канала_данных, дата, время, тревожное, значение_датчика`.

In [ ]:
import sys
import pathlib

# Подключение папки research/ к import-пути (data_utils.py)
_HERE = pathlib.Path.cwd()
if _HERE.name == "notebooks":
    RESEARCH = _HERE.parent
else:
    RESEARCH = _HERE
if str(RESEARCH) not in sys.path:
    sys.path.insert(0, str(RESEARCH))

import pandas as pd
import numpy as np
import data_utils as du

# Если True — полный пересчёт из сырых CSV (медленно, минуты).
# Если False — читать готовые артефакты из dataset/ (быстро).
RECOMPUTE = True

print("dataset:", du.find_dataset_dir())
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

# 1.1 · Схема и пример записей

Проверим заголовки всех лет и покажем пример строк. Заметьте: в 2025 заголовок повторяется в середине файла (склейка двух выгрузок) — читаем через `du.read_chunks`, который такие строки отбрасывает.

In [ ]:
# Заголовки всех лет — проверка одинаковой схемы
for fp in du.journal_files():
    with open(fp, "r", encoding="utf-8", errors="replace") as f:
        head, first = f.readline().strip(), f.readline().strip()
    print(f"{fp.name:28s} | {head} | {first[:60]}")

# Пример записей 2026 (первые 3 строки)
df = pd.read_csv(
    du.find_dataset_dir() / "extracted" / "ext-journal-2026.csv",
    dtype=str, nrows=3, on_bad_lines="skip",
)
df.head()

# 1.2 · Сводка по годам (артефакт `eda_summary.csv`)

Считаем по каждому году: число строк, уникальных каналов, диапазон дат, число тревог (флаг `t`), долю тревог, число уникальных значений датчиков.

> **Почему это важно:** объёмы данных определяют стратегию (чанки, parquet), а нестабильность доли тревог — запрет на случайный сплит (только временной).

In [ ]:
out_csv = du.find_dataset_dir() / "eda_summary.csv"
if RECOMPUTE or not out_csv.exists():
    summary_rows = []
    for fp in du.journal_files():
        year = fp.name.split("-")[2].split(".")[0]
        rows = alarms = 0
        ch_set = set()
        dates = []
        vals = set()
        for ch in du.read_chunks(fp, cols=["ид_канала_данных", "дата", "тревожное", "значение_датчика"]):
            rows += len(ch)
            ch_set.update(ch["ид_канала_данных"].unique().tolist())
            dates += [ch["дата"].min(), ch["дата"].max()]
            alarms += int((ch["тревожное"] == "t").sum())
            vals.update(ch["значение_датчика"].dropna().astype(str).unique().tolist())
        summary_rows.append({
            "год": year,
            "строк": rows,
            "уник_каналов": len(ch_set),
            "мин_дата": min(dates),
            "макс_дата": max(dates),
            "дней": (pd.to_datetime(max(dates)) - pd.to_datetime(min(dates))).days + 1,
            "тревожных": alarms,
            "доля_тревожных_%": round(100 * alarms / rows, 4),
            "уник_значений": len(vals),
        })
    summary = pd.DataFrame(summary_rows)
    summary.to_csv(out_csv, index=False, encoding="utf-8-sig")
else:
    summary = pd.read_csv(out_csv, dtype={"год": str})
    print("Прочитано из готового артефакта:", out_csv)

summary

# 1.3 · Комментарии

- **Итого ≈ 313.5 млн событий**, каналов 5.6 тыс. (2019) → 11.5 тыс. (2026). Это рост парка датчиков СМВУ.
- **Доля тревог 0.2–2.2%** — сильно несбалансированный целевой признак; в 2021 пик 2.23%.
- **2026 обрезан до 30.06** — это «будущее» для валидации прогнозов (свежие данные).
- Класс тревог **нестабилен по годам** → случайный сплит нечестен; используем **временной сплит** (train ≤ 2024, val = 2025, test = 2026).

# 1.4 · Динамика объёмов и тревог

Визуализация помогает увидеть масштаб (логика чанков/parquet) и тренды по тревожности.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
summary.plot(x="год", y="строк", kind="bar", ax=ax[0], title="Событий по годам")
summary.plot(x="год", y="доля_тревожных_%", kind="bar", ax=ax[1], title="Доля тревог %")
plt.tight_layout()
plt.show()